In [2]:
import torch
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pickle

from scipy.spatial.distance import jensenshannon
from collections import Counter

In [21]:
def js_bootstrap_threshold(
    reference_probs,
    n_samples=500,
    n_boot=2000,
    percentile=95,
    base=np.e
):
    
    reference_probs = np.asarray(reference_probs, dtype=float)
    reference_probs /= reference_probs.sum()  # normalize

    n_bins = len(reference_probs)
    distances = np.empty(n_boot)

    cdf = np.cumsum(reference_probs)

    for i in range(n_boot):
        # bootstrap draw from reference
        draws = np.searchsorted(cdf, np.random.rand(n_samples))
        counts = np.bincount(draws, minlength=n_bins)
        empirical = counts / counts.sum()

        # compute JS distance
        d = jensenshannon(empirical, reference_probs, base=base)
        distances[i] = d

    # threshold from bootstrap distribution
    threshold_dist = np.percentile(distances, percentile)
    threshold_div = threshold_dist ** 2

    return threshold_dist, threshold_div


In [22]:
with open('//Users/johnhutchens/Desktop/Practicum/Data/Wild_Dictionaries/pg2_wild_types_matrices.pickle',
           'rb') as f:
    pg_dict = pickle.load(f)

In [5]:
keys = list(pg_dict.keys())

In [32]:
j = 20
k = keys[j]

l_dist = torch.exp(pg_dict[k]['log_probs'])
l_dist
for i in range(15):
    print(i, l_dist[i].var())

0 tensor(0.0028)
1 tensor(0.0013)
2 tensor(0.0167)
3 tensor(0.0084)
4 tensor(0.0026)
5 tensor(0.0019)
6 tensor(0.0064)
7 tensor(0.0039)
8 tensor(0.0072)
9 tensor(0.0071)
10 tensor(0.0100)
11 tensor(0.0100)
12 tensor(0.0140)
13 tensor(0.0323)
14 tensor(0.0115)


In [ ]:
reference_probs = np.ones(20)/20

js_bootstrap_threshold(
    reference_probs,
    n_samples=400,
    n_boot=1000,
    percentile=95,
    base=np.e
)

(np.float64(0.09732990948480841), np.float64(0.009473111280320998))

In [49]:
boot_variances = bootstrap_variance(reference_probs, n_obs=1000, n_bootstrap=5000)
boot_variances.mean()

np.float64(1.9278578021894254e-34)

In [51]:
def bootstrap_variance_discrete_uniform(n_obs, n_bootstrap, low=1, high=20, random_seed=None):
    """
    Compute the variance of bootstrap samples from a discrete uniform distribution U(low, high).

    Parameters:
    - n_obs: number of observations per bootstrap sample
    - n_bootstrap: number of bootstrap samples
    - low: minimum integer value of the distribution (inclusive)
    - high: maximum integer value of the distribution (inclusive)
    - random_seed: for reproducibility

    Returns:
    - boot_vars: array of variances for each bootstrap sample
    """
    if random_seed is not None:
        np.random.seed(random_seed)

    ref_dist = np.arange(low, high + 1)  # discrete uniform values
    boot_vars = []

    for _ in range(n_bootstrap):
        sample = np.random.choice(ref_dist, size=n_obs, replace=True)
        boot_vars.append(np.var(sample, ddof=1))  # sample variance

    return np.array(boot_vars)

# Example usage
n_obs = 100
n_bootstrap = 1000
boot_vars = bootstrap_variance_discrete_uniform(n_obs, n_bootstrap)

print("Mean of bootstrap variances:", np.mean(boot_vars))
print("Variance of bootstrap variances:", np.var(boot_vars))


Mean of bootstrap variances: 33.19302060606061
Variance of bootstrap variances: 8.54056930699708
